# Evaluate SCALP-lite Embedding

Choose one of the registered datasets in the first code cell. The selected dataset sets the embedded input path, batch key, label key, and metrics output path.

In [ ]:
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

from scalp_lite import ScalpEstimator, score_embedding


# Choose the same dataset used in the visualization notebook.
DATASETS = {
    "zebrafish": {"input": "data/cellrank-zebrafish-scalp.h5ad", "batch_key": "Stage", "label_key": "lineages"},
    "pancreas": {"input": "data/pancreas_normalized-scalp.h5ad", "batch_key": "study", "label_key": "cell_type"},
    "pbmc3k": {"input": "data/scanpy-pbmc3k-scalp.h5ad", "batch_key": "batch", "label_key": "leiden"},
}

selected_dataset = "pancreas"
dataset = DATASETS[selected_dataset]
input_path = Path("..") / dataset["input"]
csv_path = input_path.parent / "scalp_lite_metrics.csv"
batch_key = dataset["batch_key"]
label_key = dataset["label_key"]

# Estimator is used here only for consistent data loading and key configuration.
estimator = ScalpEstimator(batch_key=batch_key, label_key=label_key)

selected_dataset, input_path, batch_key, label_key


In [ ]:
adata = estimator.to_data(input_path)
scores = score_embedding(
    adata,
    # Embedding coordinates to evaluate.
    embedding_key="X_scalp",
    # Batch column used to measure whether batches are mixed.
    batch_key=batch_key,
    # Label column used to measure whether biological labels stay coherent.
    label_key=label_key if label_key in adata.obs else None,
)
scores


In [ ]:
plot_df = scores.drop(columns=["embedding_key"], errors="ignore").melt(var_name="metric", value_name="value")
plt.figure(figsize=(9, 4))
sns.barplot(data=plot_df, x="metric", y="value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout();

In [ ]:
scores.to_csv(csv_path, index=False)
csv_path